# Schema Interconnectivity and Mapping Coverage in the Life-Sciences Linked Open Data Cloud

This notebook quantifies the structural properties of the mined schema graph and assesses how curated and inferred mappings extend cross-dataset connectivity.  

**Scope.** 
- instance-based matching
- curated SeMRA imports
- curated OLS SSSOM imports
- inference-expanded mappings 

Then measuring how each layer changes the inter-dataset reachability structure.

Three graph variants are constructed:

* **G_schema** - typed-object edges extracted from mined schemas only (no mappings).
* **G_raw** - G_schema augmented with non-inferenced mapping edges (instance + semra + sssom).
* **G_inferred** - G_raw augmented with the inferenced mapping layer.

Community structure is detected with the Louvain algorithm (NetworkX implementation) on the undirected dataset-level projection of each graph variant.

## 1. Setup

In [1]:
from __future__ import annotations

import json
import re
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from networkx.algorithms.community import louvain_communities

# ── paths ────────────────────────────────────────────────────────────────────
ROOT = Path("..").resolve()          # repo root when notebook is in notebooks/
SCHEMAS_DIR      = ROOT / "docker" / "schemas"
MAPPINGS_SSSOM   = ROOT / "docker" / "mappings" / "sssom"
MAPPINGS_SEMRA   = ROOT / "docker" / "mappings" / "semra"
MAPPINGS_INST    = ROOT / "docker" / "mappings" / "instance_matching"
MAPPINGS_INF     = ROOT / "docker" / "mappings" / "inferenced"

# ── matplotlib style ──────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})
sns.set_palette("muted")
print("Environment ready.")

Environment ready.


## 2. Artefact inventory

In [2]:
def _count_patterns(directory: Path, glob: str = "*.jsonld") -> dict:
    """Return (n_files, n_non_empty, total_pattern_count) for a mapping/schema dir."""
    files = list(directory.glob(glob))
    n_non_empty = 0
    total = 0
    for f in files:
        about = json.loads(f.read_text()).get("@about", {})
        pc = about.get("pattern_count", 0) or 0
        total += pc
        if pc > 0:
            n_non_empty += 1
    return {"files": len(files), "non_empty": n_non_empty, "patterns": total}


inventory_rows = []
for label, directory in [
    ("schemas",           SCHEMAS_DIR),
    ("sssom",             MAPPINGS_SSSOM),
    ("semra",             MAPPINGS_SEMRA),
    ("instance_matching", MAPPINGS_INST),
    ("inferenced",        MAPPINGS_INF),
]:
    glob = "**/*.jsonld" if label == "schemas" else "*.jsonld"
    stats = _count_patterns(directory, glob=glob)
    inventory_rows.append({"layer": label, **stats})

df_inv = pd.DataFrame(inventory_rows).set_index("layer")
print(df_inv.to_string())
df_inv

                   files  non_empty  patterns
layer                                        
schemas               90         82   1137574
sssom                269         97   6264658
semra                 15         15    207648
instance_matching      4          2        11
inferenced             1          1      4010


,files,non_empty,patterns
layer,,,
schemas,90,82,1137574
sssom,269,97,6264658
semra,15,15,207648
instance_matching,4,2,11
inferenced,1,1,4010


## 3. Loading utilities

Schema and mapping JSON-LD files are loaded via `MinedSchema.from_jsonld()` and
`Mapping.from_jsonld()` - the canonical rdfsolve methods that correctly expand all
CURIEs from each file's own `@context` block and reconstruct typed Pydantic objects.
Each class also provides `to_networkx()` which returns an `nx.MultiDiGraph` directly.

In [3]:
from rdfsolve.models import MinedSchema, Mapping

def _shorten(uri: str) -> str:
    """Return a compact label for a URI (last path/fragment segment)."""
    for sep in ("#", "/"):
        idx = uri.rfind(sep)
        if idx >= 0 and idx < len(uri) - 1:
            return uri[idx + 1:]
    return uri

print("rdfsolve.models imported - MinedSchema.from_jsonld / Mapping.from_jsonld available.")

rdfsolve.models imported - MinedSchema.from_jsonld / Mapping.from_jsonld available.


## 4. Load all schema edges

In [4]:
schema_files = sorted(SCHEMAS_DIR.rglob("*.jsonld"))
print(f"Schema files found: {len(schema_files)}")

schemas: list[MinedSchema] = []
for sf in schema_files:
    try:
        ms = MinedSchema.from_jsonld(sf)
        schemas.append(ms)
    except Exception as exc:
        print(f"  WARNING: could not load {sf.name}: {exc}")

# Flatten to a DataFrame for analysis
schema_rows = []
for ms in schemas:
    ds = ms.about.dataset_name or ""
    for pat in ms.patterns:
        schema_rows.append({
            "dataset":       ds,
            "source_class":  pat.subject_class,
            "predicate":     pat.property_uri,
            "target_class":  pat.object_class,
            "count":         pat.count,
            "is_literal":    pat.object_class == "Literal",
            "is_resource":   pat.object_class == "Resource",
        })

df_schema = pd.DataFrame(schema_rows)
df_sch = df_schema[~df_schema["is_literal"] & ~df_schema["is_resource"]].copy()

print(f"Schemas loaded: {len(schemas)}")
print(f"Total patterns (all types): {len(df_schema):,}")
print(f"Typed-object patterns only: {len(df_sch):,}")
print(f"Unique datasets:             {df_sch['dataset'].nunique()}")
print(f"Unique source classes:       {df_sch['source_class'].nunique():,}")

# Quick sanity check: show pattern counts per dataset (top 10)
schema_stats = (
    df_sch.groupby("dataset")
    .agg(patterns=("source_class", "count"), classes=("source_class", "nunique"))
    .sort_values("patterns", ascending=False)
)
print("\nTop 10 datasets by typed-object pattern count:")
print(schema_stats.head(10).to_string())

Schema files found: 90
Schemas loaded: 90
Total patterns (all types): 913,720
Typed-object patterns only: 264,851
Unique datasets:             59
Unique source classes:       34,398
Schemas loaded: 90
Total patterns (all types): 913,720
Typed-object patterns only: 264,851
Unique datasets:             59
Unique source classes:       34,398

Top 10 datasets by typed-object pattern count:
                   patterns  classes
dataset                             
pubchem.protein      165139    29927
mona                  88054     3609
irefindex              1326       71
cellosaurus             878       77
wikipathways.all        829       40
wikipathways            829       40
chembl                  788       62
chembl.bigcat.all       619       47
drugbank                618       50
chembl.bigcat           616       46

Top 10 datasets by typed-object pattern count:
                   patterns  classes
dataset                             
pubchem.protein      165139    29927
mona    

## 5. Load mapping edges -> build G_raw and G_inferred


In [5]:
import time
from collections import defaultdict

# ── G_schema ──────────────────────────────────────────────────────────────────
# Nodes = datasets.  Edges = at least one schema pattern whose object_class URI
# is also a subject_class in another dataset (typed-object cross-dataset links).
G_schema = nx.Graph()
for ms in schemas:
    ds = ms.about.dataset_name or ""
    G_schema.add_node(ds, pattern_count=len(ms.patterns))

class_to_datasets: dict[str, set[str]] = defaultdict(set)
for ms in schemas:
    ds = ms.about.dataset_name or ""
    for pat in ms.patterns:
        if pat.subject_class not in ("Literal", "Resource"):
            class_to_datasets[pat.subject_class].add(ds)

known_classes: set[str] = set(class_to_datasets.keys())

for ms in schemas:
    ds_src = ms.about.dataset_name or ""
    for pat in ms.patterns:
        if pat.object_class in ("Literal", "Resource"):
            continue
        for ds_tgt in class_to_datasets.get(pat.object_class, ()):
            if ds_tgt == ds_src:
                continue
            if G_schema.has_edge(ds_src, ds_tgt):
                G_schema[ds_src][ds_tgt]["weight"] += 1
            else:
                G_schema.add_edge(ds_src, ds_tgt, weight=1)

print(f"G_schema - nodes: {G_schema.number_of_nodes():3d}  edges: {G_schema.number_of_edges():5d}")
print(f"  known class URIs in schema: {len(known_classes):,}")

# ── G_raw via Mapping.dataset_graph() ────────────────────────────────────────
# Class-level resolution pass (Layer 2c):
#   SSSOM / SeMRA / instance_matching files are scanned.  For each mapping edge,
#   both endpoint URIs are expanded and looked up in class_to_datasets.
#   If BOTH match a known schema class the edge contributes a dataset-pair weight.
#

t0 = time.perf_counter()
G_raw = Mapping.dataset_graph(
    paths=(sorted(MAPPINGS_SSSOM.glob("*.jsonld"))
         + sorted(MAPPINGS_SEMRA.glob("*.jsonld"))
         + sorted(MAPPINGS_INST.glob("*.jsonld"))),
    class_to_datasets=class_to_datasets,
    base_graph=G_schema.copy(),
    strategies={"sssom_import", "semra_import", "instance_matcher"},
)
t_raw = time.perf_counter() - t0

new_raw = (set(frozenset(e) for e in G_raw.edges())
         - set(frozenset(e) for e in G_schema.edges()))
print(f"\nG_raw    - nodes: {G_raw.number_of_nodes():3d}  edges: {G_raw.number_of_edges():5d}"
      f"  ({t_raw:.1f}s)  +{len(new_raw)} new pairs over G_schema")
if new_raw:
    for fs in sorted(new_raw, key=lambda fs: -G_raw[list(fs)[0]][list(fs)[1]]["weight"]):
        u, v = tuple(fs)
        print(f"  {u} -- {v}  w={G_raw[u][v]['weight']}")
else:
    print("  (0 new pairs - instance resolver needed for SSSOM/SeMRA term-level bridging)")

# ── G_inferred via Mapping.dataset_graph() ───────────────────────────────────
t0 = time.perf_counter()
G_inferred = Mapping.dataset_graph(
    paths=sorted(MAPPINGS_INF.glob("*.jsonld")),
    class_to_datasets=class_to_datasets,
    base_graph=G_raw.copy(),
    strategies={"inferenced"},
)
t_inf = time.perf_counter() - t0

new_inf = (set(frozenset(e) for e in G_inferred.edges())
         - set(frozenset(e) for e in G_raw.edges()))
print(f"G_inferred- nodes: {G_inferred.number_of_nodes():3d}  edges: {G_inferred.number_of_edges():5d}"
      f"  ({t_inf:.1f}s)  +{len(new_inf)} new pairs over G_raw")
print(f"\nTotal scan time: {t_raw + t_inf:.1f}s")


G_schema - nodes:  88  edges:   871
  known class URIs in schema: 74,296

G_raw    - nodes:  88  edges:   871  (47.9s)  +0 new pairs over G_schema
  (0 new pairs - instance resolver needed for SSSOM/SeMRA term-level bridging)

G_raw    - nodes:  88  edges:   871  (47.9s)  +0 new pairs over G_schema
  (0 new pairs - instance resolver needed for SSSOM/SeMRA term-level bridging)
G_inferred- nodes:  88  edges:   871  (0.2s)  +0 new pairs over G_raw

Total scan time: 48.2s
G_inferred- nodes:  88  edges:   871  (0.2s)  +0 new pairs over G_raw

Total scan time: 48.2s


## 6. Mapping inventory

Distribution of mapping edges by strategy and predicate type.

In [ ]:
import ujson

_SKIP_KEYS = frozenset({"void:inDataset", "dcterms:created"})

strategy_counts:  Counter = Counter()
predicate_counts: Counter = Counter()

mapping_dirs = {
    "sssom":      MAPPINGS_SSSOM,
    "semra":      MAPPINGS_SEMRA,
    "inferenced": MAPPINGS_INF,
}

for layer, directory in mapping_dirs.items():
    for mf in sorted(directory.glob("*.jsonld")):
        try:
            raw = ujson.loads(mf.read_bytes())
            strategy = raw.get("@about", {}).get("strategy", "unknown")
            ctx = {**raw.get("@about", {}).get("curie_map") or {}, **raw.get("@context", {})}
            for node in raw.get("@graph", ()):
                for key in node:
                    if key[0] == "@" or key in _SKIP_KEYS:
                        continue
                    val = node[key]
                    targets = val if isinstance(val, list) else (val,)
                    n = sum(1 for t in targets if isinstance(t, dict) and t.get("@id"))
                    strategy_counts[strategy] += n
                    predicate_counts[key] += n
        except Exception:
            pass

if strategy_counts:
    def _shorten_pred(uri: str) -> str:
        for sep in ("#", "/"):
            idx = uri.rfind(sep)
            if idx >= 0 and idx < len(uri) - 1:
                return uri[idx + 1:]
        return uri

    strat_series = pd.Series(strategy_counts).sort_values(ascending=False)
    print("Edges per strategy:")
    print(strat_series.to_string())
    print()

    pred_short = Counter({_shorten_pred(k): v for k, v in predicate_counts.items()})
    pred_series = pd.Series(pred_short).sort_values(ascending=False)
    print(f"Distinct predicates: {len(pred_series)}")
    print(pred_series.head(20).to_string())

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    strat_series.plot(kind="bar", ax=axes[0], color="steelblue", edgecolor="white")
    axes[0].set_title("Mapping edges by strategy")
    axes[0].set_xlabel("")
    axes[0].set_ylabel("Edge count")
    axes[0].tick_params(axis="x", rotation=30)

    pred_series.head(15).plot(kind="barh", ax=axes[1], color="coral", edgecolor="white")
    axes[1].set_title("Top 15 mapping predicates")
    axes[1].set_xlabel("Edge count")
    axes[1].invert_yaxis()

    plt.tight_layout()
    plt.show()
else:
    print("No mapping edges loaded - check mapping directories.")


SyntaxError: invalid syntax (4138903824.py, line 19)

## 7. Graph construction

Three graph variants are built at the **dataset level** (nodes = datasets we mined a
schema for; edges = at least one class-to-class bridging connection).
Edge weights count the number of distinct class-pair bridges between each pair.

* **G_schema** - baseline: cross-dataset typed-object edges from mined schemas alone.
  An edge (A, B) exists when dataset A's schema references a class URI that dataset B
  also instantiates as a subject class.  This is the smallest graph.
* **G_raw** - G_schema augmented with curated mapping edges (sssom + semra).
  Only edges whose *both* endpoint datasets appear in G_schema are added.
* **G_inferred** - G_raw augmented with inferenced mapping edges (inversion,
  transitivity).  Same node-filter applies.

A fourth, commented-out variant **G_owl** would propagate predicates via subclass /
equivalence reasoning before computing cross-dataset reachability - see the commented
block below.

In [ ]:
# G_schema, G_raw, and G_inferred were all built during the streaming pass in
# the previous cell.  Nothing to do here except confirm the node counts.
print(f"G_schema - nodes: {G_schema.number_of_nodes():3d}  edges: {G_schema.number_of_edges():5d}")
print(f"G_raw    - nodes: {G_raw.number_of_nodes():3d}  edges: {G_raw.number_of_edges():5d}")
print(f"G_inferred- nodes: {G_inferred.number_of_nodes():3d}  edges: {G_inferred.number_of_edges():5d}")

# ─────────────────────────────────────────────────────────────────────────────
# [FUTURE WORK] G_owl - OWL-reasoned predicate propagation layer
#
# If class C is related to C' via subClassOf / equivalentClass / skos:exactMatch
# the cross-dataset reachability of the schema graph can be expanded.
# Left for future work: requires materialised subClassOf in inference.py.
# ─────────────────────────────────────────────────────────────────────────────


## 8. Connectivity metrics

For each graph variant: number of edges, connected components, largest component size,
diameter (on the largest component), and density.

In [ ]:
def graph_metrics(G: nx.Graph, label: str) -> dict:
    n = G.number_of_nodes()
    e = G.number_of_edges()
    components = list(nx.connected_components(G))
    n_comp = len(components)
    lcc = max(components, key=len)
    lcc_size = len(lcc)
    # Diameter of the largest connected component
    lcc_subgraph = G.subgraph(lcc)
    diameter = nx.diameter(lcc_subgraph) if lcc_size > 1 else 0
    density = nx.density(G)
    return {
        "graph":       label,
        "nodes":       n,
        "edges":       e,
        "components":  n_comp,
        "lcc_nodes":   lcc_size,
        "diameter":    diameter,
        "density":     round(density, 4),
    }


rows = [
    graph_metrics(G_schema,   "G_schema"),
    graph_metrics(G_raw,      "G_raw"),
    graph_metrics(G_inferred, "G_inferred"),
]
df_metrics = pd.DataFrame(rows).set_index("graph")
print(df_metrics.to_string())
df_metrics

## 9. Hidden bridges

Edges present in G_inferred but absent in G_raw represent dataset pairs that can only be
joined via inferred (transitive / inverse) mappings - connections invisible to analysis
based on curated mappings alone.

In [ ]:
raw_edge_set      = set(frozenset(e) for e in G_raw.edges())
inferred_edge_set = set(frozenset(e) for e in G_inferred.edges())
new_bridges       = inferred_edge_set - raw_edge_set

print(f"Edges in G_inferred not in G_raw (hidden bridges): {len(new_bridges)}")
if new_bridges:
    bridge_rows = []
    for fs in sorted(new_bridges):
        u, v = tuple(fs)
        w = G_inferred[u][v]["weight"]
        bridge_rows.append({"dataset_A": u, "dataset_B": v, "inferred_pairs": w})
    df_bridges = (
        pd.DataFrame(bridge_rows)
        .sort_values("inferred_pairs", ascending=False)
        .reset_index(drop=True)
    )
    print(df_bridges.to_string(index=False))
    df_bridges.head(20)
else:
    print("No new bridges introduced by inference over the current mapping set.")

## 10. Hub ontology analysis

Classes that appear most often as bridging nodes (in both source and target position across
mapping edges) act as integration hubs.  Betweenness centrality on the dataset graph
identifies which datasets occupy structurally critical positions.

In [ ]:
# ── Betweenness centrality on G_inferred ─────────────────────────────────────
bc     = nx.betweenness_centrality(G_inferred, weight="weight", normalized=True)
degree = dict(G_inferred.degree(weight="weight"))

df_centrality = (
    pd.DataFrame({"dataset": list(bc.keys()), "betweenness": list(bc.values())})
    .assign(degree=lambda d: d["dataset"].map(degree))
    .sort_values("betweenness", ascending=False)
    .reset_index(drop=True)
)
print("Top 20 datasets by betweenness centrality (G_inferred):")
print(df_centrality.head(20).to_string(index=False))

# ── Top bridging class URIs ───────────────────────────────────────────────────
# Derived from class_to_datasets: classes shared across the most datasets
# are the best structural hubs (schema-level proxy for bridge participation).
class_bridge_counts = Counter(
    {cls: len(ds) for cls, ds in class_to_datasets.items() if len(ds) > 1}
)
if class_bridge_counts:
    print(f"\nTop 20 classes spanning multiple datasets (schema-level hubs):")
    for cls, cnt in class_bridge_counts.most_common(20):
        print(f"  {cnt:3d} datasets  {_shorten(cls)}")


## 11. Community detection (Louvain)

Louvain community detection is applied to the undirected, weighted G_inferred graph, sort of what we see in Kamdar & Musen (2021).
We do it for the schema G, G_raw and G_inferred.
Edge weights are the number of distinct class-pair bridges between each dataset pair.
Communities are reported as sets of datasets; inter-community edge counts characterise the modularity of the LSLOD cloud.

In [ ]:
import random

random.seed(42)

# Only non-isolated nodes participate meaningfully in Louvain
G_connected = G_schema.copy()
G_connected.remove_nodes_from(list(nx.isolates(G_schema)))

communities = louvain_communities(G_connected, weight="weight", seed=42)
communities_sorted = sorted(communities, key=len, reverse=True)

# ── map each dataset to its community index ───────────────────────────────────
node_to_community: dict[str, int] = {}
for idx, comm in enumerate(communities_sorted):
    for node in comm:
        node_to_community[node] = idx

print(f"Number of Louvain communities (non-isolated nodes): {len(communities_sorted)}")
print(
    f"Modularity: {nx.community.modularity(G_connected, communities_sorted, weight='weight'):.4f}"
)
print()
for idx, comm in enumerate(communities_sorted):
    members = sorted(comm)
    print(
        f"  Community {idx:2d} ({len(comm):3d} datasets): {', '.join(members[:8])}"
        + (" ..." if len(members) > 8 else "")
    )

### Connectivity visualisation

Network layout of G_schema with nodes coloured by Louvain community and sized by
weighted degree.  Isolated datasets (no mapping connections) are shown separately.

In [ ]:
n_comm   = len(communities_sorted)
cmap     = cm.get_cmap("tab20", max(n_comm, 1))
pos      = nx.spring_layout(G_connected, weight="weight", seed=42, k=2.5)

node_colors = [cmap(node_to_community.get(n, 0)) for n in G_connected.nodes()]
node_sizes  = [
    50 + 10 * G_connected.degree(n, weight="weight")
    for n in G_connected.nodes()
]
edge_widths = [
    0.3 + 0.5 * np.log1p(G_connected[u][v].get("weight", 1))
    for u, v in G_connected.edges()
]

fig, ax = plt.subplots(figsize=(20, 20))
nx.draw_networkx_edges(
    G_connected, pos,
    width=edge_widths, alpha=0.35, edge_color="grey", ax=ax
)
nx.draw_networkx_nodes(
    G_connected, pos,
    node_size=node_sizes, node_color=node_colors, alpha=0.85, ax=ax
)
nx.draw_networkx_labels(
    G_connected, pos,
    font_size=6, font_color="black", ax=ax
)
ax.set_title(
    f"G_schema - {G_connected.number_of_nodes()} datasets, "
    f"{G_connected.number_of_edges()} edges, "
    f"{n_comm} Louvain communities",
    fontsize=12,
)
ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
random.seed(42)

# Only non-isolated nodes participate meaningfully in Louvain
G_connected = G_inferred.copy()
G_connected.remove_nodes_from(list(nx.isolates(G_inferred)))

communities = louvain_communities(G_connected, weight="weight", seed=42)
communities_sorted = sorted(communities, key=len, reverse=True)

# ── map each dataset to its community index ───────────────────────────────────
node_to_community: dict[str, int] = {}
for idx, comm in enumerate(communities_sorted):
    for node in comm:
        node_to_community[node] = idx

print(f"Number of Louvain communities (non-isolated nodes): {len(communities_sorted)}")
print(f"Modularity: {nx.community.modularity(G_connected, communities_sorted, weight='weight'):.4f}")
print()
for idx, comm in enumerate(communities_sorted):
    members = sorted(comm)
    print(f"  Community {idx:2d} ({len(comm):3d} datasets): {', '.join(members[:8])}"
          + (" ..." if len(members) > 8 else ""))

### Connectivity visualisation

Network layout of G_inferred with nodes coloured by Louvain community and sized by
weighted degree.  Isolated datasets (no mapping connections) are shown separately.

In [ ]:
n_comm   = len(communities_sorted)
cmap     = cm.get_cmap("tab20", max(n_comm, 1))
pos      = nx.spring_layout(G_connected, weight="weight", seed=42, k=2.5)

node_colors = [cmap(node_to_community.get(n, 0)) for n in G_connected.nodes()]
node_sizes  = [
    50 + 10 * G_connected.degree(n, weight="weight")
    for n in G_connected.nodes()
]
edge_widths = [
    0.3 + 0.5 * np.log1p(G_connected[u][v].get("weight", 1))
    for u, v in G_connected.edges()
]

fig, ax = plt.subplots(figsize=(20, 20))
nx.draw_networkx_edges(
    G_connected, pos,
    width=edge_widths, alpha=0.35, edge_color="grey", ax=ax
)
nx.draw_networkx_nodes(
    G_connected, pos,
    node_size=node_sizes, node_color=node_colors, alpha=0.85, ax=ax
)
nx.draw_networkx_labels(
    G_connected, pos,
    font_size=6, font_color="black", ax=ax
)
ax.set_title(
    f"G_inferred - {G_connected.number_of_nodes()} datasets, "
    f"{G_connected.number_of_edges()} edges, "
    f"{n_comm} Louvain communities",
    fontsize=12,
)
ax.axis("off")
plt.tight_layout()
plt.show()

# ── isolated datasets (no mapping connections at all) ─────────────────────────
isolated = sorted(nx.isolates(G_inferred))
print(f"\nIsolated datasets ({len(isolated)}): {', '.join(isolated)}")

## 13. Dataset × dataset reachability matrix

For G_schema, G_raw, and G_inferred: which dataset pairs are in the same connected
component?  The difference matrices show new connections introduced by each mapping layer.

In [ ]:
# Log-scaled heatmap of the edge-weight matrix for G_inferred (top-N datasets)
top_ds = df_centrality.head(min(30, len(df_centrality)))["dataset"].tolist()

if len(top_ds) > 1:
    adj = nx.to_pandas_adjacency(G_inferred.subgraph(top_ds), weight="weight").astype(float)
    adj_log = np.log1p(adj)  # log(1 + x) to handle zeros safely

    fig, ax = plt.subplots(figsize=(14, 11))
    sns.heatmap(
        adj_log, ax=ax, cmap="YlOrRd", linewidths=0.3, linecolor="white",
        xticklabels=True, yticklabels=True,
        cbar_kws={"label": "log(1 + Bridging class-pair count)"},
    )
    ax.set_title("Log-scaled cross-dataset mapping weight (top datasets by betweenness, G_inferred)", fontsize=11)
    plt.xticks(rotation=45, ha="right", fontsize=7)
    plt.yticks(rotation=0, fontsize=7)
    plt.tight_layout()
    plt.show()


## 14. Schema structural profile

- Distribution of pattern counts and class counts across the 90 mined schemas;
- Top predicates observed within schema typed-object patterns.

In [ ]:
schema_stats = []
for ms in schemas:
    schema_stats.append({
        "dataset":       ms.about.dataset_name or ms.about.endpoint or "unknown",
        "pattern_count": ms.about.pattern_count or len(ms.patterns),
        "class_count":   len({p.subject_class for p in ms.patterns}),
        "to_patterns":   sum(1 for p in ms.patterns if p.object_class not in ("Literal", "Resource")),
        "lit_patterns":  sum(1 for p in ms.patterns if p.object_class == "Literal"),
        "strategy":      ms.about.strategy,
    })

df_schema_stats = pd.DataFrame(schema_stats).sort_values("pattern_count", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# X-axis on log scale: use log-spaced bins and set xscale to 'log'
pattern_counts = df_schema_stats["pattern_count"].clip(lower=1)  # avoid zeros for log scale
log_bins = np.logspace(np.log10(pattern_counts.min()), np.log10(pattern_counts.max()), 30)

axes[0].hist(
    pattern_counts,
    bins=log_bins, color="steelblue", edgecolor="white",
)
axes[0].set_xscale("log")
axes[0].set_title("Distribution of pattern counts per dataset")
axes[0].set_xlabel("Pattern count (log scale)")
axes[0].set_ylabel("Number of datasets")
# nicer x ticks at powers of 10, but show plain numeric labels
logmin = int(np.floor(np.log10(pattern_counts.min())))
logmax = int(np.ceil(np.log10(pattern_counts.max())))
xticks = [10 ** i for i in range(logmin, logmax + 1)]
axes[0].set_xticks(xticks)

import matplotlib.ticker as mticker
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda val, pos: f"{int(val):,}"))
axes[0].tick_params(axis="x", rotation=45)

axes[1].hist(
    df_schema_stats["class_count"],
    bins=30, color="seagreen", edgecolor="white", log=False,
)
axes[1].set_title("Distribution of class counts per dataset")
axes[1].set_xlabel("Number of distinct subject classes")
axes[1].set_ylabel("Number of datasets")

plt.tight_layout()
plt.show()

print("Largest schemas by pattern count:")
print(df_schema_stats.head(15).to_string(index=False))

print("Schema predicate frequency (from loaded patterns)")
schema_preds: Counter = Counter()
for ms in schemas:
    for pat in ms.patterns:
        if pat.object_class not in ("Literal", "Resource"):
            schema_preds[_shorten(pat.property_uri)] += 1

print(f"\nTop 20 predicates in schema typed-object patterns:")
for pred, cnt in schema_preds.most_common(20):
    print(f"  {cnt:8,d}  {pred}")

## 15. Summary

Key figures for the paper
- Mapping inventory and inference expansion,
- Cross-endpoint connectivity graph.

In [ ]:
n_sssom = strategy_counts.get("sssom_import", 0)
n_semra = strategy_counts.get("semra_import", 0)
n_inf   = strategy_counts.get("inferenced", 0)

raw_edge_set      = set(frozenset(e) for e in G_raw.edges())
inferred_edge_set = set(frozenset(e) for e in G_inferred.edges())
schema_edge_set   = set(frozenset(e) for e in G_schema.edges())
new_pairs_raw = raw_edge_set - schema_edge_set
new_pairs_inf = inferred_edge_set - raw_edge_set

summary = {
    "Mined schemas":                      len(schemas),
    "Total typed-object schema patterns": len(df_sch),
    "Total schema patterns (all types)":  len(df_schema),
    "SSSOM mapping edges":                n_sssom,
    "SeMRA mapping edges":                n_semra,
    "Inferenced mapping edges":           n_inf,
    "G_schema nodes / edges":             f"{G_schema.number_of_nodes()} / {G_schema.number_of_edges()}",
    "G_raw nodes / edges":                f"{G_raw.number_of_nodes()} / {G_raw.number_of_edges()}",
    "G_inferred nodes / edges":           f"{G_inferred.number_of_nodes()} / {G_inferred.number_of_edges()}",
    "G_schema components":                df_metrics.loc["G_schema",   "components"],
    "G_raw components":                   df_metrics.loc["G_raw",      "components"],
    "G_inferred components":              df_metrics.loc["G_inferred", "components"],
    "New dataset pairs (raw maps)":       len(new_pairs_raw),
    "New dataset pairs (inference)":      len(new_pairs_inf),
    "Louvain communities (G_inferred)":   len(communities_sorted),
    "Isolated datasets":                  len(isolated),
}

for k, v in summary.items():
    print(f"  {k:<45s}: {v}")
